In [169]:
# imports

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')


%matplotlib inline
pd.options.mode.chained_assignment = None  # default='warn'

In [170]:
pd.set_option('display.max_rows', 750)
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', None)

In [171]:
ls -lh ./data

total 1,5M
-rw-r--r-- 1 mijka mijka 1,2M  5 févr. 09:37 2016_Building_Energy_Benchmarking.csv
-rw-r--r-- 1 mijka mijka 320K  5 févr. 09:52 clean_seattle_buildings.csv


In [172]:
path = './data/'
filename = 'clean_seattle_buildings.csv'
csv_path = path + filename
df = pd.read_csv(csv_path, on_bad_lines='skip', low_memory=False)

In [173]:
df.head()

,DataYear,BuildingType,PrimaryPropertyType,CouncilDistrictCode,Neighborhood,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),LargestPropertyUseType,LargestPropertyUseTypeGFA,SecondLargestPropertyUseType,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseType,ThirdLargestPropertyUseTypeGFA,SiteEnergyUseWN(kBtu),DefaultData,ComplianceStatus,TotalGHGEmissions,geometry
0,2016,NonResidential,Hotel,7,DOWNTOWN,47.61220,-122.33799,1927,1.0,12,88434,0,88434,Hotel,88434.0,No UseType,0.0,No UseType,0.0,7456910.0,False,Compliant,249.98,POINT (-122.33799 47.6122)
1,2016,NonResidential,Hotel,7,DOWNTOWN,47.61317,-122.33393,1996,1.0,11,103566,15064,88502,Hotel,83880.0,Parking,15064.0,Restaurant,4622.0,8664479.0,False,Compliant,295.86,POINT (-122.33393 47.61317)
2,2016,NonResidential,Hotel,7,DOWNTOWN,47.61393,-122.33810,1969,1.0,41,956110,196718,759392,Hotel,756493.0,No UseType,0.0,No UseType,0.0,73937112.0,False,Compliant,2089.28,POINT (-122.3381 47.61393)
3,2016,NonResidential,Hotel,7,DOWNTOWN,47.61412,-122.33664,1926,1.0,10,61320,0,61320,Hotel,61320.0,No UseType,0.0,No UseType,0.0,6946800.5,False,Compliant,286.43,POINT (-122.33664 47.61412)
4,2016,NonResidential,Hotel,7,DOWNTOWN,47.61375,-122.34047,1980,1.0,18,175580,62000,113580,Hotel,123445.0,Parking,68009.0,Swimming Pool,0.0,14656503.0,False,Compliant,505.01,POINT (-122.34047 47.61375)


In [174]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1538 entries, 0 to 1537
Data columns (total 24 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   DataYear                         1538 non-null   int64  
 1   BuildingType                     1538 non-null   object 
 2   PrimaryPropertyType              1538 non-null   object 
 3   CouncilDistrictCode              1538 non-null   int64  
 4   Neighborhood                     1538 non-null   object 
 5   Latitude                         1538 non-null   float64
 6   Longitude                        1538 non-null   float64
 7   YearBuilt                        1538 non-null   int64  
 8   NumberofBuildings                1538 non-null   float64
 9   NumberofFloors                   1538 non-null   int64  
 10  PropertyGFATotal                 1538 non-null   int64  
 11  PropertyGFAParking               1538 non-null   int64  
 12  PropertyGFABuilding(

In [175]:
df.dtypes

DataYear                             int64
BuildingType                        object
PrimaryPropertyType                 object
CouncilDistrictCode                  int64
Neighborhood                        object
Latitude                           float64
Longitude                          float64
YearBuilt                            int64
NumberofBuildings                  float64
NumberofFloors                       int64
PropertyGFATotal                     int64
PropertyGFAParking                   int64
PropertyGFABuilding(s)               int64
LargestPropertyUseType              object
LargestPropertyUseTypeGFA          float64
SecondLargestPropertyUseType        object
SecondLargestPropertyUseTypeGFA    float64
ThirdLargestPropertyUseType         object
ThirdLargestPropertyUseTypeGFA     float64
SiteEnergyUseWN(kBtu)              float64
DefaultData                           bool
ComplianceStatus                    object
TotalGHGEmissions                  float64
geometry   

In [176]:
df.shape

(1538, 24)

In [177]:
df.isna().sum().sum()

0

In [ ]:
#save separately energy star score
energy_star_score = df['ENERGYSTARScore']
df.drop('ENERGYSTARScore', axis=1, inplace=True)
df.shape

In [178]:
#df.isna().sum().sum()

# Prep Vectors & Matrices

In [179]:
y = df.copy()[{'SiteEnergyUseWN(kBtu)', 'TotalGHGEmissions'}]
X = df.copy().drop(['SiteEnergyUseWN(kBtu)', 'TotalGHGEmissions'], axis=1)

## Normalisation & One Hot Encoder

In [180]:
X.select_dtypes(['category','object']).nunique()

BuildingType                       5
PrimaryPropertyType               19
Neighborhood                      13
LargestPropertyUseType            51
SecondLargestPropertyUseType      45
ThirdLargestPropertyUseType       37
ComplianceStatus                   2
geometry                        1473
dtype: int64

In [181]:
X.shape

(1538, 22)

In [182]:
### isoler var num et var cat
categorical_columns = X.select_dtypes(['category','object']).columns
numerical_columns = X.select_dtypes(['int32','float64']).columns

In [183]:
categorical_columns

Index(['BuildingType', 'PrimaryPropertyType', 'Neighborhood',
       'LargestPropertyUseType', 'SecondLargestPropertyUseType',
       'ThirdLargestPropertyUseType', 'ComplianceStatus', 'geometry'],
      dtype='object')

In [184]:
numerical_columns

Index(['Latitude', 'Longitude', 'NumberofBuildings',
       'LargestPropertyUseTypeGFA', 'SecondLargestPropertyUseTypeGFA',
       'ThirdLargestPropertyUseTypeGFA'],
      dtype='object')

In [185]:
df.describe()

,DataYear,CouncilDistrictCode,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),LargestPropertyUseTypeGFA,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseTypeGFA,SiteEnergyUseWN(kBtu),TotalGHGEmissions
count,1538.0,1538.000000,1538.000000,1538.000000,1538.000000,1538.000000,1538.000000,1.538000e+03,1538.000000,1.538000e+03,1.538000e+03,1538.000000,1538.000000,1.538000e+03,1538.000000
mean,2016.0,4.329649,47.615026,-122.333185,1962.012354,1.138492,4.104031,1.129270e+05,13034.667750,9.989234e+04,9.356508e+04,18476.435824,2824.070480,8.293447e+06,182.011964
std,0.0,2.208087,0.048497,0.024582,32.307170,1.179278,6.688601,1.934357e+05,43008.772568,1.716701e+05,1.613423e+05,50979.994641,17523.778754,2.277765e+07,727.893139
min,2016.0,1.000000,47.499170,-122.411820,1900.000000,1.000000,0.000000,1.128500e+04,0.000000,1.092500e+04,0.000000e+00,0.000000,0.000000,5.811420e+04,-0.800000
25%,2016.0,2.000000,47.582970,-122.343225,1930.250000,1.000000,1.000000,2.938300e+04,0.000000,2.830525e+04,2.555075e+04,0.000000,0.000000,1.337961e+06,20.520000
50%,2016.0,4.000000,47.612030,-122.333000,1965.000000,1.000000,2.000000,4.872450e+04,0.000000,4.676450e+04,4.358550e+04,0.000000,0.000000,2.739967e+06,49.540000
75%,2016.0,7.000000,47.648610,-122.322230,1989.000000,1.000000,4.000000,1.036642e+05,0.000000,9.447150e+04,9.186250e+04,12471.250000,0.000000,7.198460e+06,138.177500
max,2016.0,7.000000,47.733870,-122.258640,2015.000000,27.000000,99.000000,2.200000e+06,512608.000000,2.200000e+06,1.719643e+06,639931.000000,459748.000000,4.716139e+08,16870.980000


In [186]:
df[categorical_columns].describe()

,BuildingType,PrimaryPropertyType,Neighborhood,LargestPropertyUseType,SecondLargestPropertyUseType,ThirdLargestPropertyUseType,ComplianceStatus,geometry
count,1538,1538,1538,1538,1538,1538,1538,1538
unique,5,19,13,51,45,37,2,1473
top,NonResidential,Small- and Mid-Sized Office,GREATER DUWAMISH,Office,No UseType,No UseType,Compliant,POINT (-122.29898 47.66246)
freq,1351,281,337,471,751,1228,1453,8


In [187]:
df[categorical_columns].dtypes

BuildingType                    object
PrimaryPropertyType             object
Neighborhood                    object
LargestPropertyUseType          object
SecondLargestPropertyUseType    object
ThirdLargestPropertyUseType     object
ComplianceStatus                object
geometry                        object
dtype: object

In [188]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

ohe = OneHotEncoder(sparse=False)
enc = OrdinalEncoder() 
ss = StandardScaler()


In [189]:
### normaliser num
X[numerical_columns] = ss.fit_transform(X[numerical_columns])

In [190]:
### transformer var cat (OHE ou ordinal)
## bcp valeurs diff =>  favoriser ordinal encoding (enlever les 2 cols de target)
df[categorical_columns].nunique()

BuildingType                       5
PrimaryPropertyType               19
Neighborhood                      13
LargestPropertyUseType            51
SecondLargestPropertyUseType      45
ThirdLargestPropertyUseType       37
ComplianceStatus                   2
geometry                        1473
dtype: int64

In [197]:
df['LargestPropertyUseType'].unique()

array(['Hotel', 'Police Station', 'Other - Entertainment/Public Assembly',
       'Library', 'Fitness Center/Health Club/Gym', 'Social/Meeting Hall',
       'Courthouse', 'Other', 'K-12 School', 'College/University',
       'Automobile Dealership', 'Office', 'Self-Storage Facility',
       'Non-Refrigerated Warehouse', 'Other - Mall', 'Medical Office',
       'Retail Store', 'Hospital (General Medical & Surgical)', 'Museum',
       'Repair Services (Vehicle, Shoe, Locksmith, etc)',
       'Other/Specialty Hospital', 'Financial Office',
       'Distribution Center', 'Parking', 'Worship Facility', 'Restaurant',
       'Data Center', 'Laboratory', 'Supermarket/Grocery Store',
       'Urgent Care/Clinic/Other Outpatient', 'No UseType',
       'Other - Services', 'Strip Mall', 'Wholesale Club/Supercenter',
       'Refrigerated Warehouse', 'Manufacturing/Industrial Plant',
       'Other - Recreation', 'Lifestyle Center',
       'Other - Public Services', 'Fire Station', 'Performing Arts',
  

In [ ]:
# => réencoder manuellement en classes générales

In [191]:
cat_cols_to_ohe = ['BuildingType', 'ComplianceStatus']
cat_cols_to_ord = ['PrimaryPropertyType', 'Neighborhood', 
                   'LargestPropertyUseType', 'SecondLargestPropertyUseType', 
                   'ThirdLargestPropertyUseType', 'geometry']

In [192]:
ohe.fit_transform(X[cat_cols_to_ohe])

array([[0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       [0., 1., 0., ..., 0., 1., 0.],
       ...,
       [0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 1., 0.],
       [0., 0., 1., ..., 0., 1., 0.]])

In [193]:
enc.fit_transform(X[cat_cols_to_ord])

array([[   2.,    3.,   12.,   21.,   13.,  980.],
       [   2.,    3.,   12.,   31.,   29.,  812.],
       [   2.,    3.,   12.,   21.,   13.,  990.],
       ...,
       [   9.,    7.,   29.,   11.,   35., 1269.],
       [   7.,    5.,   29.,   11.,   27.,  423.],
       [   7.,    5.,   29.,   11.,   27.,  111.]])

In [194]:
X[cat_cols_to_ord].head()

,PrimaryPropertyType,Neighborhood,LargestPropertyUseType,SecondLargestPropertyUseType,ThirdLargestPropertyUseType,geometry
0,Hotel,DOWNTOWN,Hotel,No UseType,No UseType,POINT (-122.33799 47.6122)
1,Hotel,DOWNTOWN,Hotel,Parking,Restaurant,POINT (-122.33393 47.61317)
2,Hotel,DOWNTOWN,Hotel,No UseType,No UseType,POINT (-122.3381 47.61393)
3,Hotel,DOWNTOWN,Hotel,No UseType,No UseType,POINT (-122.33664 47.61412)
4,Hotel,DOWNTOWN,Hotel,Parking,Swimming Pool,POINT (-122.34047 47.61375)


In [195]:
X[cat_cols_to_ohe].head()

,BuildingType,ComplianceStatus
0,NonResidential,Compliant
1,NonResidential,Compliant
2,NonResidential,Compliant
3,NonResidential,Compliant
4,NonResidential,Compliant


In [196]:
X[cat_cols_to_ord] = enc.fit_transform(X[cat_cols_to_ord])

In [131]:
X.head()

,DataYear,BuildingType,PrimaryPropertyType,CouncilDistrictCode,Neighborhood,Latitude,Longitude,YearBuilt,NumberofBuildings,NumberofFloors,PropertyGFATotal,PropertyGFAParking,PropertyGFABuilding(s),LargestPropertyUseType,LargestPropertyUseTypeGFA,SecondLargestPropertyUseType,SecondLargestPropertyUseTypeGFA,ThirdLargestPropertyUseType,ThirdLargestPropertyUseTypeGFA,DefaultData,ComplianceStatus,geometry
0,2016,NonResidential,2.0,7,3.0,-0.058296,-0.195525,1927,-0.117476,12,88434,0,88434,12.0,-0.031813,21.0,-0.362543,13.0,-0.161209,False,Compliant,980.0
1,2016,NonResidential,2.0,7,3.0,-0.038288,-0.030313,1996,-0.117476,11,103566,15064,88502,12.0,-0.060048,31.0,-0.066959,29.0,0.102633,False,Compliant,812.0
2,2016,NonResidential,2.0,7,3.0,-0.022612,-0.200001,1969,-0.117476,41,956110,196718,759392,12.0,4.110166,21.0,-0.362543,13.0,-0.161209,False,Compliant,990.0
3,2016,NonResidential,2.0,7,3.0,-0.018693,-0.140590,1926,-0.117476,10,61320,0,61320,12.0,-0.199920,21.0,-0.362543,13.0,-0.161209,False,Compliant,938.0
4,2016,NonResidential,2.0,7,3.0,-0.026325,-0.296443,1980,-0.117476,18,175580,62000,113580,12.0,0.185256,31.0,0.971924,35.0,-0.161209,False,Compliant,1041.0


In [168]:
X = pd.merge(X[cat_cols_to_ohe], 
          pd.DataFrame(columns = ohe.get_feature_names().tolist(),
              data = ohe.fit_transform(X[cat_cols_to_ohe])),
        left_index = True, right_index = True)

In [138]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1538 entries, 0 to 1537
Data columns (total 9 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   BuildingType                     1538 non-null   object 
 1   ComplianceStatus                 1538 non-null   object 
 2   x0_Campus                        1538 non-null   float64
 3   x0_NonResidential                1538 non-null   float64
 4   x0_Nonresidential COS            1538 non-null   float64
 5   x0_Nonresidential WA             1538 non-null   float64
 6   x0_SPS-District K-12             1538 non-null   float64
 7   x1_Compliant                     1538 non-null   float64
 8   x1_Error - Correct Default Data  1538 non-null   float64
dtypes: float64(7), object(2)
memory usage: 108.3+ KB


In [135]:
X.head()

,BuildingType,ComplianceStatus,x0_Campus,x0_NonResidential,x0_Nonresidential COS,x0_Nonresidential WA,x0_SPS-District K-12,x1_Compliant,x1_Error - Correct Default Data
0,NonResidential,Compliant,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,NonResidential,Compliant,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,NonResidential,Compliant,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3,NonResidential,Compliant,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,NonResidential,Compliant,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [199]:
# 3 preds différentes (regression...mini une linéaire et une non-linéaire ex: linear regressor, random forest)
# optimisation hyper paramètres

## comparer avec même score (RMSE, R² (surtout pour linéaires))
## logger + comparer temps de test/entraînement
## ressortir tableau comparatif
## choisir meilleur

## une fois modele choisi, ajouter energystarscore
## evaluer si cela améliore ou non la modélisation
## utiliser librairie pour évaluer => librairie SHAP


In [ ]:
## possibilité de lister différents scorings à calculer en une fois => doc

In [ ]:
## essayer assembliste / random forest / svr / linearregression 
## check doc scikitlab regresseurs
## /!\ ne pas oublier normalisation

In [ ]:
## /!\ 2 targets => virer les 2 => 2 colonnes de moins pour l'ordinal encoding puis ajouter une target à la fois par training

In [ ]:
# analyse
# nettoyage (outliers + valeurs vides)
# (penser à encoder null en une autre valeur si besoin, par ex pour l'encoding, puis signifier à l'inputer que telle valeur est nulle)
# encoding
# encode > cat > -5 : OHE
#                +5 : ordinal
#         > cat > 